# Exercise 3.3: Table Modeling and Write Optimization

Building on the partitioning and sort order concepts from E1.2, this exercise focuses on write-time optimization, specifically how Spark's **distribution modes** affect file layout and write performance. Distribution modes are the primary new concept here; the partition and sort order experiments serve as quick validation that those concepts still apply when distribution modes are added to the mix.

In this exercise, you'll learn how to optimize Apache Iceberg tables using the NYC Yellow Taxi dataset:
- **Distribution modes**: Control how Spark distributes data before writing (the main focus of this exercise)
- **Partition strategies**: Quick comparison of partition granularities and their write-time costs (building on E1.2)
- **Sort orders**: Write-cost analysis for sort orders (building on E1.2)

## Learning Objectives
- Compare unpartitioned, daily, and monthly partitioning
- Implement and test sort orders
- Understand distribution mode tradeoffs
- Measure the impact of modeling decisions on query performance

> **Note on benchmarks:** The configurations in this notebook were chosen to illustrate real-world tradeoffs between different modeling strategies. However, absolute timings will vary based on your local environment (CPU, memory, disk speed, Docker resource allocation). In production with distributed clusters and larger datasets, the performance differences are typically more pronounced.

⚠️ **IMPORTANT**: This environment uses **Spark Connect** — a single Spark server runs in the background and all notebooks connect to it as lightweight clients (no per-notebook JVM). If you see `ConnectionRefusedError: [Errno 111] Connection refused` when initializing the Spark Session, the Spark Connect server may not be running. Check the Docker container logs with `docker logs jupyter-spark`. If the server failed to start, try restarting the container with `docker compose restart jupyter`.

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("TableModeling") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.modeling")
print("Namespace 'modeling' created!")

## Download NYC Taxi Data

We'll use NYC Yellow Taxi trip data for **June through September 2023** (~200 MB total) to compare different table modeling strategies. A larger dataset makes performance differences between strategies more visible.

In [ ]:
import boto3
from botocore.client import Config
import urllib.request
import os

s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket = "warehouse"

for month in range(6, 10):
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    key = f"raw/{filename}"
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        print(f"{filename} already in MinIO, skipping download")
    except:
        local_path = f"/tmp/{filename}"
        print(f"Downloading {filename} (~50MB)...")
        urllib.request.urlretrieve(base_url.format(month), local_path)
        s3_client.upload_file(local_path, bucket, key)
        os.remove(local_path)
        print(f"  Uploaded to s3a://{bucket}/{key}")

print("\nTaxi data ready in MinIO!")

In [ ]:
taxi_df = spark.read.parquet(
    *[f"s3a://warehouse/raw/yellow_tripdata_2023-{m:02d}.parquet" for m in range(6, 10)]
)
taxi_df.cache()
taxi_df.createOrReplaceTempView("taxi_raw")
count = taxi_df.count()
print(f"Loaded {count:,} taxi trips (June-September 2023) for testing")
taxi_df.show(5)

## Part 1: Partition Granularity

### Strategy 1: Unpartitioned Table

In [ ]:
print("Creating UNPARTITIONED table...")
start = time.time()

taxi_df.writeTo("polaris.modeling.taxi_unpartitioned") \
    .using("iceberg") \
    .tableProperty("write.target-file-size-bytes", "134217728") \
    .createOrReplace()

write_time_unpart = time.time() - start
print(f"Write time: {write_time_unpart:.2f} seconds")

In [ ]:
print("Unpartitioned file statistics:")
spark.sql("""
    SELECT COUNT(*) as file_count,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
    FROM polaris.modeling.taxi_unpartitioned.files
""").show()

### Strategy 2: Daily Partitioning

In [ ]:
print("Creating DAILY PARTITIONED table...")
start = time.time()

spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_daily")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_daily
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES ('write.target-file-size-bytes' = '134217728')
    AS SELECT * FROM taxi_raw
""")

write_time_daily = time.time() - start
print(f"Write time: {write_time_daily:.2f} seconds")

In [ ]:
print("Daily partition file statistics:")
spark.sql("""
    SELECT COUNT(*) as file_count,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           COUNT(DISTINCT partition) as partition_count
    FROM polaris.modeling.taxi_daily.files
""").show()

### Strategy 3: Monthly Partitioning

In [ ]:
print("Creating MONTHLY PARTITIONED table...")
start = time.time()

spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_monthly")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_monthly
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES ('write.target-file-size-bytes' = '134217728')
    AS SELECT * FROM taxi_raw
""")

write_time_monthly = time.time() - start
print(f"Write time: {write_time_monthly:.2f} seconds")

In [ ]:
print("Monthly partition file statistics:")
spark.sql("""
    SELECT COUNT(*) as file_count,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           COUNT(DISTINCT partition) as partition_count
    FROM polaris.modeling.taxi_monthly.files
""").show()

### Compare Query Performance

In [ ]:
query = """
    SELECT COUNT(*) as count, ROUND(AVG(fare_amount), 2) as avg_fare
    FROM {table}
    WHERE tpep_pickup_datetime >= TIMESTAMP '2023-06-15 00:00:00'
      AND tpep_pickup_datetime < TIMESTAMP '2023-06-16 00:00:00'
"""

print("Query: All trips on June 15, 2023")
print("=" * 60)

for name, table in [("Unpartitioned", "polaris.modeling.taxi_unpartitioned"),
                     ("Daily", "polaris.modeling.taxi_daily"),
                     ("Monthly", "polaris.modeling.taxi_monthly")]:
    start = time.time()
    result = spark.sql(query.format(table=table)).collect()
    elapsed = time.time() - start
    print(f"{name:15s}: {elapsed:.3f}s  (count: {result[0]['count']:,}, avg_fare: {result[0]['avg_fare']})")

In [ ]:
query2 = """
    SELECT COUNT(*) as count
    FROM {table}
    WHERE tpep_pickup_datetime >= TIMESTAMP '2023-06-15 14:00:00'
      AND tpep_pickup_datetime < TIMESTAMP '2023-06-15 15:00:00'
"""

print("\nQuery: Trips on June 15, 2-3 PM")
print("=" * 60)

for name, table in [("Unpartitioned", "polaris.modeling.taxi_unpartitioned"),
                     ("Daily", "polaris.modeling.taxi_daily"),
                     ("Monthly", "polaris.modeling.taxi_monthly")]:
    start = time.time()
    count = spark.sql(query2.format(table=table)).collect()[0]['count']
    elapsed = time.time() - start
    print(f"{name:15s}: {elapsed:.3f}s  (count: {count:,})")

### Partition Granularity Summary

In [ ]:
print("=" * 60)
print("PARTITION STRATEGY COMPARISON")
print("=" * 60)
print(f"{'Strategy':<20} {'Write Time':<12}")
print("-" * 40)
print(f"{'Unpartitioned':<20} {write_time_unpart:>10.2f}s")
print(f"{'Daily':<20} {write_time_daily:>10.2f}s")
print(f"{'Monthly':<20} {write_time_monthly:>10.2f}s")
print("=" * 60)

**Observations:**
- **Monthly:** Fewest files, but day-range queries must scan full month files
- **Daily:** Good balance with one file per day, efficient for day-range queries
- **Unpartitioned:** Simplest, but scans all data for any time filter

### Try It: Test with a Different Query Pattern

The benchmarks above used a single-day filter. Try a broader query (e.g., full-month scan) or a narrower one (e.g., a single hour). How do the partition strategies compare when the query aligns perfectly with the partition granularity vs when it doesn't?

In [ ]:
# my_query = """
#     SELECT COUNT(*), ROUND(AVG(total_amount), 2)
#     FROM {table}
#     WHERE tpep_pickup_datetime >= TIMESTAMP '2023-06-15 14:00:00'
#       AND tpep_pickup_datetime <  TIMESTAMP '2023-06-15 15:00:00'
# """
#
# for name, table in [("Unpartitioned", "polaris.modeling.taxi_unpartitioned"),
#                      ("Daily", "polaris.modeling.taxi_daily"),
#                      ("Monthly", "polaris.modeling.taxi_monthly")]:
#     start = time.time()
#     spark.sql(my_query.format(table=table)).collect()
#     elapsed = time.time() - start
#     print(f"{name:<20} {elapsed:.3f}s")

In [ ]:
for t in ['taxi_unpartitioned', 'taxi_daily', 'taxi_monthly']:
    spark.sql(f"DROP TABLE IF EXISTS polaris.modeling.{t}")
print("Part 1 tables cleaned up!")

## Part 2: Sort Orders

### Table Without Sort Order

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_unsorted")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_unsorted
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT * FROM taxi_raw
""")

print("Unsorted table created!")

### Table With Sort Order

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_sorted")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_sorted
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT * FROM taxi_raw WHERE 1=0
""")

spark.sql("""
    ALTER TABLE polaris.modeling.taxi_sorted
    WRITE ORDERED BY PULocationID
""")

taxi_df.writeTo("polaris.modeling.taxi_sorted").append()

print("Sorted table created (sorted by PULocationID)!")

### Compare Query Performance

In [ ]:
query = """
    SELECT COUNT(*) as count, ROUND(AVG(fare_amount), 2) as avg_fare
    FROM {table}
    WHERE PULocationID = 132
      AND tpep_pickup_datetime >= '2023-06-15' AND tpep_pickup_datetime < '2023-06-16'
"""

print("Query: Trips from location 132 on June 15")
print("=" * 60)

start = time.time()
r1 = spark.sql(query.format(table="polaris.modeling.taxi_unsorted")).collect()
unsorted_time = time.time() - start
print(f"Unsorted: {unsorted_time:.3f}s  (count: {r1[0]['count']}, avg_fare: {r1[0]['avg_fare']})")

start = time.time()
r2 = spark.sql(query.format(table="polaris.modeling.taxi_sorted")).collect()
sorted_time = time.time() - start
print(f"Sorted:   {sorted_time:.3f}s  (count: {r2[0]['count']}, avg_fare: {r2[0]['avg_fare']})")

if unsorted_time > sorted_time:
    print(f"\nSort order gave {unsorted_time/sorted_time:.1f}x speedup on filtered query")
else:
    print(f"\nResults similar at this data volume")

### Try It: Try a Different Sort Key

The sorted table above uses `PULocationID`. Try creating a table sorted by `fare_amount` or `trip_distance` instead. Run a query that filters on your sort key and compare it to the unsorted version. Does the sort order help more for high-selectivity filters?

In [ ]:
# my_sort_key = "fare_amount"  # or trip_distance, DOLocationID, etc.
# my_filter = "fare_amount > 100"  # match your sort key for best effect

# spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_my_sort")
# spark.sql("""
#     CREATE TABLE polaris.modeling.taxi_my_sort
#     USING iceberg
#     PARTITIONED BY (days(tpep_pickup_datetime))
#     AS SELECT * FROM taxi_raw
#     WHERE 1=0
# """)
# spark.sql(f"ALTER TABLE polaris.modeling.taxi_my_sort WRITE ORDERED BY {my_sort_key}")
# taxi_df.writeTo("polaris.modeling.taxi_my_sort").append()
#
# start = time.time()
# spark.sql(f"SELECT COUNT(*) FROM polaris.modeling.taxi_my_sort WHERE {my_filter}").collect()
# print(f"Sorted by {my_sort_key}: {time.time() - start:.3f}s")
#
# spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_my_sort")

In [ ]:
for t in ['taxi_unsorted', 'taxi_sorted']:
    spark.sql(f"DROP TABLE IF EXISTS polaris.modeling.{t}")
print("Part 2 tables cleaned up!")

## Part 3: Distribution Modes

Distribution mode is a Spark-specific Iceberg setting that controls whether and how Spark rearranges (shuffles) data across its worker tasks before writing. A shuffle is like sorting mail into bins before delivery. Data is sent across the network so that all rows for the same partition end up on the same worker. This costs network I/O but produces cleaner files. Iceberg supports three distribution modes:

- **hash**: Shuffles data by partition value so each task writes to one partition. This is the default for tables without a sort order.
- **range**: Performs a range-based shuffle that both groups by partition AND sorts within each partition. This is the default for tables with a sort order.
- **none**: Skips the shuffle entirely. Faster if the data is already organized, but risky for small file generation if it is not.

Whether distribution mode matters depends on the relationship between your data's natural ordering and your partition key. We'll run two experiments to demonstrate this. First, let's verify that our raw parquet data is naturally ordered by pickup date.

### Raw Data is Already Ordered by Day

The NYC taxi parquet files are produced one per month, with rows in chronological order. This means the data is naturally clustered by day without any shuffling. Let's verify this.

In [ ]:
from pyspark.sql.functions import to_date, min as spark_min, max as spark_max

print("Date range per source parquet file:")
for m in range(6, 10):
    path = f"s3a://warehouse/raw/yellow_tripdata_2023-{m:02d}.parquet"
    stats = spark.read.parquet(path) \
        .filter("tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-10-01'") \
        .agg(
            spark_min(to_date("tpep_pickup_datetime")).alias("first"),
            spark_max(to_date("tpep_pickup_datetime")).alias("last"),
            F.count("*").alias("n")
        ).collect()[0]
    print(f"  {path.split('/')[-1]}: {stats.first} to {stats.last}  ({stats.n:,} rows)")

In [ ]:
print("Trips per day across the dataset (first and last 5 days shown):")
print("The data follows a smooth chronological progression.\n")

daily = taxi_df \
    .filter("tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-10-01'") \
    .withColumn("pickup_date", to_date("tpep_pickup_datetime")) \
    .groupBy("pickup_date") \
    .agg(F.count("*").alias("trip_count")) \
    .orderBy("pickup_date")

total_days = daily.count()
print(f"Total distinct days: {total_days}\n")

print("First 5 days:")
daily.limit(5).show(truncate=False)
print("Last 5 days:")
daily.orderBy(F.desc("pickup_date")).limit(5).orderBy("pickup_date").show(truncate=False)

print("The data spans a contiguous range of dates, confirming the parquet files")
print("are organized chronologically. This natural ordering matters for distribution modes.")

Each input partition spans at most a few days, and they arrive in chronological order. This has a direct impact on distribution mode behavior: when data is already grouped by the partition key, the shuffle is unnecessary. When it is not, the shuffle is essential.

We'll demonstrate this with two experiments:
- **Experiment A:** Partition by `bucket(16, PULocationID)` where the data is **not** sorted by this column, so distribution mode makes a big difference.
- **Experiment B:** Partition by `days(tpep_pickup_datetime)` where the data **is** already sorted by day, so hash and none produce nearly identical results.

### Experiment A: Bucket Partitioning (Data Not Pre-Sorted)

The `bucket(N, col)` transform hashes column values into N fixed groups. It's useful for columns with high cardinality where temporal transforms don't apply. Choose N based on your target file count. More buckets means more files but finer-grained pruning.

When partitioning by a column the data is not organized by, distribution mode controls whether Spark shuffles data before writing. We'll compare all three modes.

#### Hash Distribution (Default Without Sort Order)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_dist_hash")

print("Writing with HASH distribution mode (bucketed by PULocationID)...")
start = time.time()

spark.sql("""
    CREATE TABLE polaris.modeling.taxi_dist_hash
    USING iceberg
    PARTITIONED BY (bucket(16, PULocationID))
    TBLPROPERTIES ('write.distribution-mode' = 'hash')
    AS SELECT * FROM taxi_raw
""")

hash_time = time.time() - start
print(f"Write time: {hash_time:.2f} seconds")

In [ ]:
print("Hash distribution file stats:")
spark.sql("""
    SELECT COUNT(*) as total_files, COUNT(DISTINCT partition) as partitions,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
    FROM polaris.modeling.taxi_dist_hash.files
""").show()

#### Range Distribution (Default With Sort Order)

When a table has a sort order, Iceberg's Spark integration automatically uses **range** distribution. Range distribution performs a more sophisticated shuffle: it reads a small sample of the data to estimate value distributions and determine partition boundaries (this adds a small overhead compared to hash), then distributes rows so that each task writes to one partition with its data already sorted. This produces both clean partitioning and sorted files in a single pass.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_dist_range")

spark.sql("""
    CREATE TABLE polaris.modeling.taxi_dist_range
    USING iceberg
    PARTITIONED BY (bucket(16, PULocationID))
    AS SELECT * FROM taxi_raw WHERE 1=0
""")

spark.sql("""
    ALTER TABLE polaris.modeling.taxi_dist_range
    WRITE ORDERED BY fare_amount
""")

print("Writing with RANGE distribution mode (bucketed + sorted by fare_amount)...")
start = time.time()

taxi_df.writeTo("polaris.modeling.taxi_dist_range").append()

range_time = time.time() - start
print(f"Write time: {range_time:.2f} seconds")

In [ ]:
print("Range distribution file stats:")
spark.sql("""
    SELECT COUNT(*) as total_files, COUNT(DISTINCT partition) as partitions,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
    FROM polaris.modeling.taxi_dist_range.files
""").show()

#### None Distribution (No Shuffle)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_dist_none")

print("Writing with NONE distribution mode (bucketed by PULocationID)...")
start = time.time()

spark.sql("""
    CREATE TABLE polaris.modeling.taxi_dist_none
    USING iceberg
    PARTITIONED BY (bucket(16, PULocationID))
    TBLPROPERTIES ('write.distribution-mode' = 'none')
    AS SELECT * FROM taxi_raw
""")

none_time = time.time() - start
print(f"Write time: {none_time:.2f} seconds")

In [ ]:
print("None distribution file stats:")
spark.sql("""
    SELECT COUNT(*) as total_files, COUNT(DISTINCT partition) as partitions,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb
    FROM polaris.modeling.taxi_dist_none.files
""").show()

#### Experiment A Summary

In [ ]:
hash_stats = spark.sql("SELECT COUNT(*) as f FROM polaris.modeling.taxi_dist_hash.files").collect()[0].f
range_stats = spark.sql("SELECT COUNT(*) as f FROM polaris.modeling.taxi_dist_range.files").collect()[0].f
none_stats = spark.sql("SELECT COUNT(*) as f FROM polaris.modeling.taxi_dist_none.files").collect()[0].f

print("=" * 80)
print("EXPERIMENT A: bucket(16, PULocationID), data NOT pre-sorted")
print("=" * 80)
print(f"{'Mode':<12} {'Sort Order':<14} {'Time':<8} {'Files':<8} {'Files/Partition':<18} {'Notes'}")
print("-" * 80)
print(f"{'Hash':<12} {'--':<14} {hash_time:>5.1f}s  {hash_stats:>5}   {hash_stats//16:>5}              {'Clean layout'}")
range_fpp = f"~{range_stats // 16}"
print(f"{'Range':<12} {'fare_amount':<14} {range_time:>5.1f}s  {range_stats:>5}   {range_fpp:>5}              {'Clean + sorted'}")
print(f"{'None':<12} {'--':<14} {none_time:>5.1f}s  {none_stats:>5}   {none_stats//16:>5}              {'Small file problem!'}")
print("=" * 80)

When data is **not** pre-sorted by the partition key, distribution mode makes a dramatic difference:

- **Hash:** Shuffle by partition value ensures each task writes to exactly one bucket, producing clean, uniform files.
- **Range:** Range-based shuffle also groups by partition, but additionally sorts within each partition. Slightly slower and may produce a few extra files from size splits, but the sorted layout benefits read-time queries on the sort column.
- **None:** No shuffle, so each task writes to every bucket its data touches. With PULocationID scattered randomly, this creates 8 files per bucket: a textbook small file problem.

### Experiment B: Day Partitioning (Data Already Pre-Sorted)

We showed above that the raw data is already in chronological order. When we partition by `days(tpep_pickup_datetime)`, the data naturally groups by partition key. In this case, does the shuffle actually help? Let's compare hash and none with daily partitioning.

#### Hash Distribution (Daily)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_dist_hash_day")

print("Writing with HASH distribution mode (daily partition)...")
start = time.time()

spark.sql("""
    CREATE TABLE polaris.modeling.taxi_dist_hash_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES ('write.distribution-mode' = 'hash')
    AS SELECT * FROM taxi_raw
""")

hash_day_time = time.time() - start
print(f"Write time: {hash_day_time:.2f} seconds")

In [ ]:
print("Hash distribution (daily) file stats:")
spark.sql("""
    SELECT COUNT(*) as total_files, COUNT(DISTINCT partition) as partitions,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb
    FROM polaris.modeling.taxi_dist_hash_day.files
""").show()

#### None Distribution (Daily)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_dist_none_day")

print("Writing with NONE distribution mode (daily partition, data is pre-sorted)...")
start = time.time()

spark.sql("""
    CREATE TABLE polaris.modeling.taxi_dist_none_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES ('write.distribution-mode' = 'none')
    AS SELECT * FROM taxi_raw
""")

none_day_time = time.time() - start
print(f"Write time: {none_day_time:.2f} seconds")

In [ ]:
print("None distribution (daily, pre-sorted) file stats:")
spark.sql("""
    SELECT COUNT(*) as total_files, COUNT(DISTINCT partition) as partitions,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb
    FROM polaris.modeling.taxi_dist_none_day.files
""").show()

#### Experiment B Summary

In [ ]:
hash_day_stats = spark.sql("SELECT COUNT(*) as f FROM polaris.modeling.taxi_dist_hash_day.files").collect()[0].f
noneday_stats = spark.sql("SELECT COUNT(*) as f FROM polaris.modeling.taxi_dist_none_day.files").collect()[0].f
hash_day_parts = spark.sql("SELECT COUNT(DISTINCT partition) as p FROM polaris.modeling.taxi_dist_hash_day.files").collect()[0].p
noneday_parts = spark.sql("SELECT COUNT(DISTINCT partition) as p FROM polaris.modeling.taxi_dist_none_day.files").collect()[0].p

print("=" * 70)
print("EXPERIMENT B: days(tpep_pickup_datetime), data IS pre-sorted")
print("=" * 70)
print(f"{'Mode':<12} {'Time':<8} {'Files':<8} {'Partitions':<12} {'Files/Partition':<18}")
print("-" * 70)
hash_day_fpp = f"{hash_day_stats / hash_day_parts:.1f}"
none_day_fpp = f"{noneday_stats / noneday_parts:.1f}"
print(f"{'Hash':<12} {hash_day_time:>5.1f}s  {hash_day_stats:>5}   {hash_day_parts:>8}     {hash_day_fpp:>5}")
print(f"{'None':<12} {none_day_time:>5.1f}s  {noneday_stats:>5}   {noneday_parts:>8}     {none_day_fpp:>5}")
print("=" * 70)


When data **is** already ordered by the partition key, both hash and none produce well-partitioned files because no partition mixing occurs in either case. None produces more files than hash (200 vs 129) because boundary partitions get split across Spark tasks. Each task that spans a day boundary writes a small fragment to each day. But the file *contents* are well-partitioned, so query performance is similar.

**Bottom line:** Check whether your data's natural ordering matches your partition key. If it does, 
`none` avoids unnecessary shuffle overhead (though it may produce slightly more files at partition boundaries). If it does not (as in Experiment A), `hash` or `range` is essential.

### Sort Order Degradation Over Multiple Writes

A freshly written sorted table has non-overlapping file ranges. For example, if sorted by `fare_amount`, file 1 might cover $0-$15, file 2 $15-$30, and so on. A query like `WHERE fare_amount BETWEEN 50 AND 55` can skip most files entirely using column min/max metadata.

But each new write (INSERT, MERGE, etc.) creates its **own** set of sorted files. These new files cover the full `fare_amount` range, overlapping with files from previous writes. After many writes, every value range has multiple files covering it, and min/max pruning degrades.

To demonstrate this clearly, we'll insert the same month of data repeatedly and measure how query performance changes as overlapping files accumulate.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_sort_decay")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_sort_decay
    USING iceberg
    TBLPROPERTIES ('write.target-file-size-bytes' = '8388608')
    AS SELECT * FROM taxi_raw WHERE 1=0
""")
spark.sql("ALTER TABLE polaris.modeling.taxi_sort_decay WRITE ORDERED BY fare_amount")

june_df = taxi_df.filter("tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-07-01'")
june_df.createOrReplaceTempView("june_trips")
june_count = june_df.count()
print(f"Created unpartitioned table sorted by fare_amount (8 MB target file size)")
print(f"Using June data ({june_count:,} rows per write). We'll insert it repeatedly")

In [ ]:
benchmark_query = """
    SELECT COUNT(*), ROUND(AVG(tip_amount), 2)
    FROM polaris.modeling.taxi_sort_decay
    WHERE fare_amount BETWEEN 50 AND 55
"""

overlap_query = """
    SELECT COUNT(*) FROM polaris.modeling.taxi_sort_decay.files
    WHERE readable_metrics.fare_amount.lower_bound <= 55
      AND readable_metrics.fare_amount.upper_bound >= 50
"""

def write_and_benchmark(label):
    file_count = spark.sql(
        "SELECT COUNT(*) FROM polaris.modeling.taxi_sort_decay.files"
    ).collect()[0][0]
    overlap = spark.sql(overlap_query).collect()[0][0]
    t = time.time()
    spark.sql(benchmark_query).collect()
    elapsed = time.time() - t
    print(f"{label:<22} {file_count:>6}      {overlap:>6} / {file_count:<6}    {elapsed:.3f}s")
    return file_count, overlap, elapsed

checkpoints = [1, 10]
current_writes = 0

print("Inserting June data repeatedly and benchmarking...\n")
print(f"{'State':<22} {'Files':<10} {'Matching $50-$55':<18} {'Query Time'}")
print("-" * 68)

for target in checkpoints:
    while current_writes < target:
        spark.sql("INSERT INTO polaris.modeling.taxi_sort_decay SELECT * FROM june_trips")
        current_writes += 1
    write_and_benchmark(f"After {current_writes} writes")

Each of the 10 writes created its own set of sorted files. Every write batch has a file covering $50-$55, a file covering $10-$15, and so on. That means 10 files now overlap with any given fare_amount query, one from each write.

Now let's compact all 10 batches into a single globally sorted set of files using `rewrite_data_files`.

In [ ]:
print("Compacting with sort order...")
start = time.time()

result = spark.sql("""
    CALL polaris.system.rewrite_data_files(
        table => 'modeling.taxi_sort_decay',
        strategy => 'sort'
    )
""").collect()[0]

compact_time = time.time() - start
print(f"Compaction took {compact_time:.1f}s")
print(f"Rewrote {result['rewritten_data_files_count']} files into {result['added_data_files_count']} files")
print()
write_and_benchmark("After compaction")


**Why this happens:** Each INSERT creates its own independently sorted files. After the first write, there is exactly one file covering any given fare_amount range. After 10 writes, there are 10 files covering that range, one from each write batch. A query filtering on `fare_amount` must scan all of them because min/max pruning cannot eliminate files whose ranges overlap with the predicate.

`rewrite_data_files` with `strategy => 'sort'` tells compaction to re-sort files according to the table's configured sort order, producing a single set of files with non-overlapping ranges. The default strategy (`'binpack'`) only combines small files without re-sorting. This is faster but doesn't restore sort order benefits. That's what E3.2's compaction was doing. After sort compaction, the query is back to scanning just the files that overlap the predicate range. This is similar to running `OPTIMIZE` in other table formats.

In production, you would schedule periodic `rewrite_data_files` calls (e.g., nightly or after several ingestion cycles) to consolidate overlapping files and keep sort order benefits intact.

### Try It: Combine Strategies

Create a table that combines partitioning, sort order, and distribution mode. For example: daily partitioning + sorted by `PULocationID` + hash distribution. Write data to it, then compare query performance against the tables from the previous experiments. What combination works best for your query pattern?

In [ ]:
# spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_combined")
# spark.sql("""
#     CREATE TABLE polaris.modeling.taxi_combined
#     USING iceberg
#     PARTITIONED BY (days(tpep_pickup_datetime))
#     TBLPROPERTIES ('write.distribution-mode' = 'hash')
#     AS SELECT * FROM taxi_raw
#     WHERE 1=0
# """)
# spark.sql("ALTER TABLE polaris.modeling.taxi_combined WRITE ORDERED BY PULocationID")
# taxi_df.writeTo("polaris.modeling.taxi_combined").append()
#
# start = time.time()
# spark.sql("""
#     SELECT COUNT(*), ROUND(AVG(fare_amount), 2)
#     FROM polaris.modeling.taxi_combined
#     WHERE PULocationID = 132
#       AND tpep_pickup_datetime >= '2023-06-15' AND tpep_pickup_datetime < '2023-06-16'
# """).collect()
# print(f"Combined strategy: {time.time() - start:.3f}s")
#
# spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_combined")

## Cleanup

In [ ]:
taxi_df.unpersist()
for t in ['taxi_dist_hash', 'taxi_dist_range', 'taxi_dist_none',
          'taxi_dist_hash_day', 'taxi_dist_none_day', 'taxi_sort_decay']:
    spark.sql(f"DROP TABLE IF EXISTS polaris.modeling.{t}")
print("Cleanup complete!")